# 07_SBERT_WikiSec_Base_Table4.ipynb

Reproduce **Table 4** from the Sentence-BERT paper.

This notebook reproduces **SBERT-WikiSec-base**.

## Paper configuration

- Backbone: `bert-base-uncased`
- Architecture: Siamese SBERT (BERT + MEAN pooling, shared weights)
- Pooling: Mean
- Loss: Triplet Loss, Euclidean distance, margin epsilon = 1
- Dataset: Wikipedia Section Triplets (Dor et al., 2018)
- Train for **1 epoch**
- Evaluate using **accuracy**: is positive closer to anchor than negative?

**Target score**

| Model | Paper |
|---|---:|
| SBERT-WikiSec-base | **0.8042** |

### What was fixed vs. the original notebook
1. **AttributeError: 'Series' object has no attribute 'anchor'** — the dataset used `row.anchor` (attribute access), which breaks silently if your CSV's column names don't literally match `anchor`/`positive`/`negative` (different case, whitespace, or a different schema entirely). The loader below now normalizes column names and fails loudly with a clear message if it can't find the right columns, instead of crashing deep inside the DataLoader worker.
2. **Loss computed in fp16** — `criterion(ea, ep, en)` was called *inside* the `autocast` block, so the triplet-margin loss itself ran in fp16, which is numerically fragile. Loss is now computed in fp32, outside autocast.
3. Minor: dev set is now actually used to monitor training (optional, cheap sanity check).

In [1]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "bert-base-uncased"

DATA_DIR = Path("..") / "data" / "WikiSec"

# NOTE: the wikipedia-sections-triplets.zip release from Dor et al. names
# the dev split "validation.csv", not "dev.csv". Row counts confirm this
# is the right dataset: test.csv has 222,957 rows, matching the paper's
# "222,957 test triplets" exactly; train.csv has ~1.78M rows, matching
# "about 1.8 Million training triplets".
TRAIN_FILE = DATA_DIR / "train.csv"
DEV_FILE = DATA_DIR / "validation.csv"
TEST_FILE = DATA_DIR / "test.csv"

OUTPUT_DIR = Path("outputs") / "sbert_wikisec_base"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LENGTH = 128
PHYSICAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
EFFECTIVE_BATCH_SIZE = PHYSICAL_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
EPOCHS = 1
LEARNING_RATE = 2e-5
EVAL_BATCH_SIZE = 32
USE_AMP = torch.cuda.is_available()

print("Effective batch size:", EFFECTIVE_BATCH_SIZE)

f:\Programs\PYTHON_310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
f:\Programs\PYTHON_310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
Effective batch size: 16


## Dataset layout

Extract `wikipedia-sections-triplets.zip` (Dor et al., 2018) here:

```text
data/
└── WikiSec/
    ├── train.csv        (1,779,417 rows)
    ├── validation.csv    (220,400 rows - used as the dev/monitoring split)
    └── test.csv          (222,957 rows - matches the paper exactly)
```

Native columns are `Article Title, Sentence1, Sentence2, Sentence3, Article Link`.
The loader below maps `Sentence1`→anchor, `Sentence2`→positive, `Sentence3`→negative
automatically (see `COLUMN_ALIASES`), and is tolerant of other naming
conventions too in case you swap datasets later.

In [2]:
# --- Robust column resolution -------------------------------------------
# The original notebook crashed with:
#   AttributeError: 'Series' object has no attribute 'anchor'
# because it used row.anchor (attribute access) and assumed the CSV's
# columns are literally named anchor/positive/negative. Any mismatch in
# case, whitespace, or naming convention breaks that silently until the
# DataLoader worker throws deep inside tqdm/torch internals.
#
# Fix: normalize headers, try common aliases, and fail LOUDLY with a
# useful message (listing the actual columns found) if resolution fails,
# instead of crashing later with an opaque AttributeError.

# The wikipedia-sections-triplets.zip release (Dor et al. 2018) uses
# columns: 'Article Title', 'Sentence1', 'Sentence2', 'Sentence3', 'Article Link'.
# Per its Readme: Sentence1 & Sentence2 are from the same section (anchor,
# positive); Sentence3 is from a different section (negative).
COLUMN_ALIASES = {
    "anchor": ["anchor", "anchor_text", "sentence1", "query", "a"],
    "positive": ["positive", "positive_text", "pos", "sentence2", "p"],
    "negative": ["negative", "negative_text", "neg", "sentence3", "n"],
}

def resolve_columns(df, path):
    normalized = {c: c.strip().lower() for c in df.columns}
    resolved = {}

    for canonical, aliases in COLUMN_ALIASES.items():
        match = None
        for orig_col, norm_col in normalized.items():
            if norm_col in aliases:
                match = orig_col
                break
        if match is None:
            raise ValueError(
                f"Could not find a '{canonical}' column in {path}.\n"
                f"Columns found: {list(df.columns)}\n"
                f"Expected one of: {COLUMN_ALIASES[canonical]}\n"
                f"-> Add the real column name to COLUMN_ALIASES['{canonical}'] above and rerun."
            )
        resolved[canonical] = match

    out = df[[resolved["anchor"], resolved["positive"], resolved["negative"]]].copy()
    out.columns = ["anchor", "positive", "negative"]

    # Drop rows with any missing/empty text - a common source of
    # downstream tokenizer crashes if left in.
    out = out.dropna(subset=["anchor", "positive", "negative"])
    for col in ["anchor", "positive", "negative"]:
        out[col] = out[col].astype(str)
    out = out[(out["anchor"].str.strip() != "") &
              (out["positive"].str.strip() != "") &
              (out["negative"].str.strip() != "")]

    return out.reset_index(drop=True)


# low_memory=False avoids dtype-guessing warnings on these large,
# free-text CSVs (train.csv alone is ~850MB / 1.78M rows).
raw_train_df = pd.read_csv(TRAIN_FILE, low_memory=False)
raw_dev_df = pd.read_csv(DEV_FILE, low_memory=False)
raw_test_df = pd.read_csv(TEST_FILE, low_memory=False)

print("Raw train columns:", list(raw_train_df.columns))

train_df = resolve_columns(raw_train_df, TRAIN_FILE)
dev_df = resolve_columns(raw_dev_df, DEV_FILE)
test_df = resolve_columns(raw_test_df, TEST_FILE)

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)
print("Test:", test_df.shape)

display(train_df.head())

Raw train columns: ['Article Title', 'Sentence1', 'Sentence2', 'Sentence3', 'Article Link']
Train: (1779417, 3)
Dev: (220400, 3)
Test: (222957, 3)


,anchor,positive,negative
0,Bailey was educated at Ipswich School (1972-79...,He won the Cricket Society's Wetherell Award i...,Bailey was a Fellow of Gonville and Caius Coll...
1,The record design and production was done by I...,VIS Idoli was also released as a double casset...,"Promotional video was recorded for ""Devojko ma..."
2,"Promotional video was recorded for ""Devojko ma...","""Dok dobuje kiša (u ritmu tam-tama)"" and ""Male...",The record design and production was done by I...
3,The Project 131 site is located in the Gaoqiao...,"It is about 15 km east of Xianning urban area,...",The tunnel system is reported to be 456 meters...
4,"McCline grew up in Harlem, New York City, livi...",McCline spent five years in prison for gun run...,"On March 8, 2008 McCline faced former two-time..."


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class WikiSecDataset(Dataset):
    def __init__(self, df):
        # Store as plain python lists - avoids any pandas attribute /
        # Series-indexing surprises entirely, and is faster for __getitem__.
        self.anchor = df["anchor"].tolist()
        self.positive = df["positive"].tolist()
        self.negative = df["negative"].tolist()

    def __len__(self):
        return len(self.anchor)

    def __getitem__(self, idx):
        return self.anchor[idx], self.positive[idx], self.negative[idx]


def collate_fn(batch):
    a = [x[0] for x in batch]
    p = [x[1] for x in batch]
    n = [x[2] for x in batch]

    def tok(text):
        return tokenizer(
            text,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

    return tok(a), tok(p), tok(n)

In [4]:
class SBERTWikiSec(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)

    @staticmethod
    def mean_pool(hidden, mask):
        mask = mask.unsqueeze(-1).float()
        summed = (hidden.float() * mask).sum(1)
        counts = mask.sum(1).clamp(min=1e-9)
        return summed / counts

    def encode(self, tokens):
        out = self.bert(**tokens)
        emb = self.mean_pool(out.last_hidden_state, tokens["attention_mask"])
        # NOTE: embeddings are NOT L2-normalized here. The paper's triplet
        # objective uses raw Euclidean distance between pooled embeddings
        # (Sec 3, margin epsilon=1 in Euclidean space). Normalizing first
        # would change the geometry the margin is defined over.
        return emb

    def forward(self, a, p, n):
        return self.encode(a), self.encode(p), self.encode(n)

In [5]:
train_loader = DataLoader(
    WikiSecDataset(train_df),
    batch_size=PHYSICAL_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
    num_workers=0,
)

dev_loader = DataLoader(
    WikiSecDataset(dev_df),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    WikiSecDataset(test_df),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)

model = SBERTWikiSec().to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

# Paper: Euclidean distance metric, margin epsilon = 1 (Sec 3, "Triplet Objective Function")
criterion = nn.TripletMarginLoss(margin=1.0, p=2)

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

W0925 22:17:33.607000 15536 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [6]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    for a, p, n in loader:
        a = {k: v.to(DEVICE) for k, v in a.items()}
        p = {k: v.to(DEVICE) for k, v in p.items()}
        n = {k: v.to(DEVICE) for k, v in n.items()}

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            ea, ep, en = model(a, p, n)

        # Accuracy metric per paper: is positive closer to anchor than negative?
        # Distances computed in fp32 for numerical stability.
        dist_pos = torch.norm(ea.float() - ep.float(), p=2, dim=1)
        dist_neg = torch.norm(ea.float() - en.float(), p=2, dim=1)

        correct += (dist_pos < dist_neg).sum().item()
        total += len(dist_pos)

    return correct / total

In [7]:
optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}")

    for step, (a, p, n) in enumerate(progress):
        a = {k: v.to(DEVICE) for k, v in a.items()}
        p = {k: v.to(DEVICE) for k, v in p.items()}
        n = {k: v.to(DEVICE) for k, v in n.items()}

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            ea, ep, en = model(a, p, n)

        # FIX: loss computed in fp32, outside autocast. The original
        # notebook called criterion() inside the autocast block, which
        # runs the triplet-margin loss itself in fp16 - fragile for a
        # margin-based loss where small distance differences matter.
        loss = criterion(ea.float(), ep.float(), en.float())
        running_loss += loss.item()
        loss = loss / GRADIENT_ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        progress.set_postfix(loss=f"{running_loss / (step + 1):.4f}")

    dev_acc = evaluate(model, dev_loader)
    print(f"Epoch {epoch + 1}/{EPOCHS} | Train loss: {running_loss / len(train_loader):.4f} | Dev accuracy: {dev_acc:.4f}")

print("Training complete.")

Epoch 1: 100%|██████████| 444855/444855 [17:43:36<00:00,  6.97it/s, loss=0.5017]   


Epoch 1/1 | Train loss: 0.5017 | Dev accuracy: 0.7898
Training complete.


In [8]:
accuracy = evaluate(model, test_loader)
print("Test Accuracy:", accuracy)

Test Accuracy: 0.7900088357844787


In [9]:
comparison = pd.DataFrame({
    "Model": ["SBERT-WikiSec-base"],
    "Paper": [0.8042],
    "Ours": [accuracy]
})

comparison["Difference"] = comparison["Ours"] - comparison["Paper"]

display(comparison)

comparison.to_csv(
    OUTPUT_DIR / "table4_comparison.csv",
    index=False
)

print("Saved to:", OUTPUT_DIR.resolve())

,Model,Paper,Ours,Difference
0,SBERT-WikiSec-base,0.8042,0.790009,-0.014191


Saved to: F:\SEM-7\NLP\Project\notebooks\outputs\sbert_wikisec_base
